# 원하는 포즈로 이미지 만들기
참조 사진의 자세(관절)를 뽑아서, 같은 자세의 다른 인물을 Stable Diffusion 1.5 + OpenPose ControlNet으로 만드는 튜토리얼입니다.

**순서:** 1) 설치 → 2) 모델 로드 → 3) 참조 사진 준비 → 4) 포즈 추출 → 5) 이미지 생성 → 6) 저장 → 7) 실험 기록

**왜 FLUX.2-klein 4B가 아닌가:** 처음에는 FLUX.2-klein 4B에 뼈대 이미지를 참조 이미지로 넣으려 했습니다. 하지만 FLUX.2-klein 4B용 공식 ControlNet은 확인하지 못했고, 무료 Colab(RAM 12.7GB)에서는 모델(약 16GB)을 올리는 도중 "RAM을 모두 사용해 세션이 다운"되어 이미지를 만들지 못했습니다. 과제가 "FLUX.2-klein 4B 또는 비슷한 모델"을 허용하므로, 포즈 조건용으로 만들어진 ControlNet이 있는 Stable Diffusion 1.5로 바꿨습니다.

**준비:** 메뉴 `런타임 > 런타임 유형 변경`에서 GPU(T4)를 고르세요.

In [ ]:
# [셀 1] 이 셀이 하는 일: 필요한 라이브러리를 설치한다.
# diffusers는 모델 카드 안내대로 GitHub 최신판을 설치한다. controlnet_aux는 OpenPose(포즈 추출기)를 쓰기 위한 것이다.
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate sentencepiece controlnet_aux

In [ ]:
# [셀 2] 이 셀이 하는 일: 기본 모델(Stable Diffusion 1.5)과 포즈 ControlNet을 불러온다. (처음에는 내려받느라 몇 분 걸린다)
# ControlNet: 뼈대 이미지를 조건으로 받아 같은 자세의 그림을 만들게 하는 보조 신경망이다.
# safety_checker=None: 모델 카드 예시처럼 안전 검사기를 껐다. 라이선스 조건(CreativeML OpenRAIL-M)을 지켜서 쓴다.
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

device = "cuda"
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-openpose", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    controlnet=controlnet, safety_checker=None, torch_dtype=torch.float16,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)

### 참조 사진 준비
아래 셀은 사진 파일을 올리는 창을 띄웁니다. **사람 전신이 보이는 사진 한 장**을 올리세요. 올린 사진은 직접 찍었거나 이용 허락이 있는 것만 쓰세요.

In [ ]:
# [셀 3] 이 셀이 하는 일: 참조 사진을 올리고 화면에 보여 준다.
from google.colab import files
from PIL import Image

uploaded = files.upload()                 # 파일 선택 창이 뜬다
ref_path = list(uploaded.keys())[0]       # 올린 첫 번째 파일 이름
ref_image = Image.open(ref_path).convert("RGB")
ref_image.thumbnail((1024, 1024))         # 너무 큰 사진은 줄인다
ref_image

In [ ]:
# [셀 4] 이 셀이 하는 일: 참조 사진에서 OpenPose로 관절 뼈대 이미지를 뽑는다. 결과가 pose_image에 담긴다.
from controlnet_aux import OpenposeDetector

openpose = OpenposeDetector.from_pretrained("lllyasviel/Annotators")
pose_image = openpose(ref_image, detect_resolution=512, image_resolution=1024)
pose_image

In [ ]:
# [셀 5] 이 셀이 하는 일: 뼈대 이미지 + 프롬프트로 새 이미지를 만든다.
# 프롬프트에서 바꿀 부분은 인물·장면 설명뿐이다. 자세는 뼈대 이미지가 정한다.
prompt = "a worker in a yellow helmet standing in a factory"
seed = 0

pose_small = pose_image.resize((512, 512))   # Stable Diffusion 1.5의 기본 크기에 맞춘다

result = pipe(
    prompt,
    pose_small,                        # 뼈대 이미지를 ControlNet 조건으로 넣는다
    num_inference_steps=20,            # 모델 카드 예시 값
    generator=torch.Generator(device=device).manual_seed(seed),
).images[0]
result

In [ ]:
# [셀 6] 이 셀이 하는 일: 포즈 이미지와 결과를 파일로 저장한다. 번호(01)는 실험마다 바꾼다.
pose_image.save("pose_01.png")
result.save("output_01.png")
files.download("pose_01.png")
files.download("output_01.png")
# 받은 파일을 GitHub 저장소의 samples/ 폴더에 올린다.

## 7. 실험 기록

조건: Stable Diffusion 1.5 + OpenPose ControlNet, 20스텝, 시드 0으로 고정, Colab T4. 시드는 하나만 써 봤기 때문에 아래 관찰이 다른 시드에서도 같다고는 말할 수 없습니다.

**실험 A. 같은 포즈, 프롬프트만 바꾸기** — 셀 5의 `prompt`만 바꿔 같은 `seed`로 돌렸습니다.
- 바꾼 프롬프트: `a worker in a yellow helmet standing in a factory` → `a young woman in a red dress standing in a park`
- 달라진 것: 사람, 옷, 배경이 전부 바뀌었습니다. 안전모를 쓴 작업자가 공장에서 붉은 드레스를 입은 여성으로 공원에 서 있는 모습이 됐습니다.
- 자세는 따라왔나: 네. 다리를 넓게 벌리고 몸을 기울인 자세, 한쪽 팔을 옆으로 뻗은 자세, 반대쪽 팔을 굽힌 자세가 두 결과 모두 비슷했습니다.

**실험 B. 같은 프롬프트, 다른 포즈 사진 넣기** — 같은 프롬프트(`a worker in a yellow helmet standing in a factory`)로 다른 사진의 뼈대를 넣었습니다.
- 넣은 사진(포즈): 모델 카드(`lllyasviel/sd-controlnet-openpose`)의 예시 사진 `images/pose.png`. 서 있는 사람이 팔을 굽혀 허리 앞에서 손을 모으고 고개를 옆으로 돌린 자세입니다. 이 사진은 상체 위주라 뼈대에서 다리 일부가 잘렸습니다. 셀 3에서 올리는 대신 주소로 불러왔고, 셀 4와 같은 방식으로 뼈대를 뽑았습니다.
- 달라진 것: 전신이 아니라 허리 위쪽만 나오는 구도가 됐습니다. 작업자가 조끼를 입고 책상 앞에서 손을 모은 모습입니다.
- 자세는 따라왔나: 팔을 굽혀 손을 앞에서 모은 것, 몸을 세운 것, 고개를 옆으로 돌린 것은 따라왔습니다. 다리는 화면에 나오지 않았습니다.
- 한 번 실패한 것: 처음에는 사진을 뼈대로 바꾸지 않고 ControlNet에 바로 넣었습니다. 그러자 자세와 상관없는 이미지(공장 속에 작게 서 있는 사람)가 나왔습니다. 조건 이미지는 반드시 OpenPose로 뽑은 뼈대여야 했습니다.

**정리**
- 어느 쪽을 바꿨을 때 결과가 크게 달랐나: 둘 다 크게 달랐는데 바뀐 것이 달랐습니다. 프롬프트를 바꾸면 누구인지, 옷, 배경이 바뀌고 자세는 거의 그대로였습니다. 포즈를 바꾸면 몸의 모양과 구도(전신인지 상체인지)가 바뀌고, 안전모·조끼·공장 같은 프롬프트의 내용은 유지됐습니다.
- 잘 안 따라온 부분(손, 발 등): 첫 번째 결과(포즈 1)에서 한쪽 팔이 참조와 다르게 뻗었습니다. 손가락은 손끝이 붙어 뭉개져 보였습니다. 첫 번째 결과의 얼굴은 일그러져 보였습니다. 뼈대에서 잘린 부분(다리)은 결과에도 나오지 않았습니다.